# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ikramkhan-gif1/FlyRank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I will use Logistic Regression because this is a binary classification problem where the goal is to predict whether a content item is declining.

Logistic Regression is a suitable first ML model because it is simple, interpretable, and produces probabilities that can be evaluated using ROC-AUC. It also provides a transparent comparison against the Week-4 rule-based baseline without adding unnecessary complexity.

The goal is not to use the most complex model, but to determine whether a simple ML model provides useful predictive signal beyond the existing baseline.

In [81]:
import os
import pandas as pd

REPO_PATH = "/content/FlyRank-ML-Internship"

if not os.path.exists(REPO_PATH):
    !git clone https://github.com/Ikramkhan-gif1/FlyRank-ML-Internship.git

DATA_PATH = os.path.join(
    REPO_PATH,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Loaded successfully.")


Dataset shape: (30000, 44)
Loaded successfully.


In [82]:
# Create the target for ML-08

df["impression_change_pct"] = (
    (
        df["impressions_last_30d"]
        - df["impressions_prev_30d"]
    )
    / df["impressions_prev_30d"].replace(0, pd.NA)
) * 100

df["is_declining_label"] = (
    df["impression_change_pct"] <= -20
).astype(int)

print("Target created successfully.")

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget proportions:")
print(df["is_declining_label"].value_counts(normalize=True).round(4))

Target created successfully.

Target distribution:
is_declining_label
1    16305
0    13695
Name: count, dtype: int64

Target proportions:
is_declining_label
1    0.5435
0    0.4565
Name: proportion, dtype: float64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a client-grouped split, keeping each client entirely in either the training or testing set. This prevents observations from the same client appearing in both sets and provides a more honest test of whether the model can generalize to unseen clients.

The test set is kept separate until final evaluation.

In [83]:
# Create a client-grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        df["is_declining_label"],
        groups=df["client_id"]
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("Training clients:", train_df["client_id"].nunique())
print("Testing clients:", test_df["client_id"].nunique())

shared_clients = set(train_df["client_id"]) & set(test_df["client_id"])

print("Clients shared between train and test:", len(shared_clients))


Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Clients shared between train and test: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [84]:
# Select features for Logistic Regression

features = [
    "search_volume",
    "impressions_90d",
    "days_with_impressions",
    "impressions_last_30d",
    "impressions_prev_30d",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

target = "is_declining_label"

X_train = train_df[features].copy()
X_test = test_df[features].copy()

y_train = train_df[target].copy()
y_test = test_df[target].copy()

print("Features:", features)

print("\nTraining feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

print("\nMissing values in training:")
print(X_train.isna().sum())

Features: ['search_volume', 'impressions_90d', 'days_with_impressions', 'impressions_last_30d', 'impressions_prev_30d', 'days_since_last_update', 'ctr', 'avg_position']

Training feature shape: (23837, 8)
Testing feature shape: (6163, 8)

Missing values in training:
search_volume             2319
impressions_90d              0
days_with_impressions        0
impressions_last_30d         0
impressions_prev_30d         0
days_since_last_update       0
ctr                          0
avg_position                 0
dtype: int64


In [85]:
# Train Logistic Regression

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


In [86]:
# Evaluate Logistic Regression using ROC-AUC

from sklearn.metrics import roc_auc_score

y_test_proba = model.predict_proba(X_test)[:, 1]

model_auc = roc_auc_score(
    y_test,
    y_test_proba
)

print("Logistic Regression ROC-AUC:", round(model_auc, 4))

Logistic Regression ROC-AUC: 0.8531


In [87]:
# Recreate the Week-4 baseline on the test set

test_baseline = test_df.copy()

# Staleness score
test_baseline["stale_score"] = 0

test_baseline.loc[
    test_baseline["days_since_last_update"] >= 91,
    "stale_score"
] = 1

test_baseline.loc[
    test_baseline["days_since_last_update"] >= 180,
    "stale_score"
] = 2

# Recent impression change
test_baseline["impression_change_pct"] = (
    (
        test_baseline["impressions_last_30d"]
        - test_baseline["impressions_prev_30d"]
    )
    / test_baseline["impressions_prev_30d"].replace(0, pd.NA)
) * 100

# Decline score
test_baseline["decline_score"] = (
    test_baseline["impression_change_pct"] <= -20
).astype(int) * 2

# Final baseline score
test_baseline["baseline_score"] = (
    test_baseline["stale_score"]
    + test_baseline["decline_score"]
)

print("Baseline score distribution:")
print(
    test_baseline["baseline_score"]
    .value_counts()
    .sort_index()
)

Baseline score distribution:
baseline_score
0    2349
1     648
2    2596
3     570
Name: count, dtype: int64


In [88]:
# Calculate baseline ROC-AUC and compare methods

baseline_auc = roc_auc_score(
    y_test,
    test_baseline["baseline_score"]
)

comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Logistic Regression"
    ],
    "ROC-AUC": [
        baseline_auc,
        model_auc
    ]
})

comparison["ROC-AUC"] = comparison["ROC-AUC"].round(4)

print(comparison)

                Method  ROC-AUC
0      Week-4 Baseline   1.0000
1  Logistic Regression   0.8531


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

I will examine where Logistic Regression makes incorrect predictions and which features have the strongest influence on its predictions. This helps explain the model rather than judging it only by its ROC-AUC.

In [89]:
# Step 9 — Error analysis

# Convert predicted probabilities into 0/1 predictions
y_test_pred = (y_test_proba >= 0.5).astype(int)

# Create an error-analysis table
error_analysis = test_df.copy()

error_analysis["actual"] = y_test.values
error_analysis["predicted"] = y_test_pred
error_analysis["prediction_probability"] = y_test_proba

# Keep only incorrect predictions
errors = error_analysis[
    error_analysis["actual"] != error_analysis["predicted"]
].copy()

print("Total test rows:", len(test_df))
print("Incorrect predictions:", len(errors))

print(
    "Error rate:",
    round(len(errors) / len(test_df), 4)
)

print("\nActual vs predicted:")
print(
    pd.crosstab(
        error_analysis["actual"],
        error_analysis["predicted"]
    )
)


Total test rows: 6163
Incorrect predictions: 1577
Error rate: 0.2559

Actual vs predicted:
predicted     0     1
actual               
0          2510   487
1          1090  2076


In [90]:
# Step 10 — Interpret Logistic Regression features

coefficients = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.named_steps["classifier"].coef_[0]
})

coefficients["Absolute_coefficient"] = (
    coefficients["Coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "Absolute_coefficient",
    ascending=False
)

print(coefficients)

                  Feature  Coefficient  Absolute_coefficient
3    impressions_last_30d   -36.679769             36.679769
4    impressions_prev_30d    30.269692             30.269692
1         impressions_90d     1.286581              1.286581
2   days_with_impressions     0.416028              0.416028
5  days_since_last_update     0.136665              0.136665
6                     ctr    -0.108312              0.108312
7            avg_position    -0.104677              0.104677
0           search_volume    -0.013887              0.013887


### Interpretation

The Logistic Regression model achieved a ROC-AUC of 0.8531 on the held-out test set. It made 1,577 incorrect predictions out of 6,163 test rows, giving an observed error rate of 25.59%.

The confusion matrix shows 487 false positives and 1,090 false negatives. The model therefore missed more declining items than it incorrectly flagged as declining.

The model coefficients show that `impressions_last_30d` and `impressions_prev_30d` are the strongest signals in the model. Their large coefficient magnitudes indicate that recent impression levels contribute most strongly to the model's predictions.

The Week-4 baseline achieved a ROC-AUC of 1.0000, while Logistic Regression achieved 0.8531 on the same held-out test set. Therefore, the simpler baseline performed better in this experiment.

This result should be interpreted carefully because the target was constructed using the same 20% impression-decline rule used by the baseline. The perfect baseline score is therefore expected from the target definition and should not be interpreted as evidence that the baseline is generally perfect.

Overall, the Logistic Regression model provides an interpretable ML comparison and identifies the main predictive signals, but the observed results do not justify replacing the simpler Week-4 baseline with the model.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.